# 1. Context 

Notebook to assess sanity of results obtained by surya ocr model

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import sys

In [ ]:
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

# 2. Imports

In [ ]:
notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))


In [5]:
from src.evaluation.metrics import cer, wer
from src.common.constants import language_code_norm_map, language_to_writing_system
from src.common.utils import get_script_results

# 3. Utils

In [6]:
# get csv paths for all language
results_root = Path("../results/google")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [8]:
writing_sys_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [24]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result, result_root=results_root)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

# 4. Computing WER & CER

## 4.1. UTC Normalisation

For language supprted by `indic-nlp-library` are normalised, where as languages not supported are left as it is

In [ ]:
def normalize_text(text: str, lang: str, norm_factory: IndicNormalizerFactory):
    """Normalize text for a given language using indic normalizer"""
    if lang not in language_code_norm_map:
        return text
    code = language_code_norm_map[lang]
    normalizer = norm_factory.get_normalizer(code)
    normalized_text = normalizer.normalize(text)
    return normalized_text

In [12]:
norm_factory = IndicNormalizerFactory()

In [25]:
cols_to_norm = ['ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3']

for col in cols_to_norm:
    consolidated_df[col] = consolidated_df.apply(lambda x: normalize_text(x[col], x['language'], norm_factory), axis=1)

## 4.1. Estimate Error Rates

In [26]:
# computing cer & wer
gt_col = 'ground_truth'
ocr_output_cols = ['ocr_output_L_0', 'ocr_output_L_1', 'ocr_output_L_2', 'ocr_output_L_3']
cols_create_cer = ['cer_l0', 'cer_l1', 'cer_l2', 'cer_l3']
cols_create_wer = ['wer_l0', 'wer_l1', 'wer_l2', 'wer_l3']
for ocr_output_lvl, col_crt_cer in dict(zip(ocr_output_cols, cols_create_cer)).items():
    consolidated_df[col_crt_cer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: cer(x[gt_col], x[ocr_output_lvl]), axis=1)
for ocr_output_lvl, col_crt_wer in dict(zip(ocr_output_cols, cols_create_wer)).items():
    consolidated_df[col_crt_wer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: wer(x[gt_col], x[ocr_output_lvl]), axis=1)

In [27]:
agg_results = (consolidated_df.groupby(['script', 'language']).agg(CER_AVG_L0=('cer_l0', 'median'),
                                                    CER_AVG_L1=('cer_l1', 'median'),
                                                    CER_AVG_L2=('cer_l2', 'median'),
                                                    CER_AVG_L3=('cer_l3', 'median'),
                                                    WER_AVG_L0=('wer_l0', 'median'),
                                                    WER_AVG_L1=('wer_l1', 'median'),
                                                    WER_AVG_L2=('wer_l2', 'median'),
                                                    WER_AVG_L3=('wer_l3', 'median')
                                                    ).round(3))

In [28]:
agg_results.columns = agg_results.columns.str.upper()

In [29]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

In [22]:
agg_results.to_clipboard()

In [32]:
consolidated_df.round(3).to_clipboard(index=False)